# 🚀 TV5 / TOÀN NHÓM — BENCHMARK & CRAWLING MS1 TRÊN COLAB CPU-ONLY

**Dự án:** arXiv Data Science Pipeline (Intro to Data Science — HCMUS) — **Nhóm 6**  
**Yêu cầu đề tài:** *"The testbed for the final benchmarking is a Google Colab instance running on CPU-only mode."*  
**Cơ chế lưu trữ an toàn:** Tự động kết nối **Google Drive** để bảo toàn 100% dữ liệu và checkpoint (`state/stages`, `state/discovery`, `corpus/`), giúp khôi phục (`--resume`) mượt mà nếu phiên Colab bị ngắt.

---
### 📋 BẢNG PHÂN CHIA DẢI ID NHÓM 6 (`configs/ms1_group6_ranges.json`)
| Thành viên | Vai trò | Dải ID được giao | Số lượng | Thư mục lưu Google Drive |
| :--- | :--- | :--- | :---: | :--- |
| **TV1** | Leader & Discovery | `2508.12872:2508.17871` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv1/` |
| **TV2** | Source Downloader | `2506.15788:2506.20787` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv2/` |
| **TV3** | File Extractor | `2411.08799:2411.13798` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv3/` |
| **TV4** | Metadata & S2 Refs | `2410.18025:2410.23024` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv4/` |
| **TV5** | QA & Report | `2410.03025:2410.08024` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv5/` |

> ⚠️ **HƯỚNG DẪN CHẠY ĐỘC LẬP:**  
> Mỗi dải 5.000 bài tốn khoảng 10-11 tiếng. **5 thành viên mở 5 tài khoản Colab độc lập để chạy song song dải của mình**. Cả nhóm sẽ hoàn thành toàn bộ 25.000 bài chỉ trong 1 đêm!

## 1. Gắn kết Google Drive (Bắt buộc để lưu Checkpoint bền vững)
> Dữ liệu tải về và checkpoint sẽ được lưu trực tiếp trên Google Drive. Khi Colab bị timeout sau vài tiếng, bạn chỉ cần mở lại notebook và chạy tiếp, hệ thống sẽ **tiếp tục ngay bài đang dở mà không bị mất dữ liệu**.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/arxiv_ms1_data')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print('✅ Google Drive đã kết nối thành công!')
print(f'📁 Thư mục gốc lưu trữ: {DRIVE_BASE}')

## 2. Kiểm tra Môi trường phần cứng (Bắt buộc CPU-only)
> Đề tài yêu cầu benchmark ở chế độ CPU-only. Cell này sẽ kiểm tra xem máy có lỡ bật GPU hay không.

In [ ]:
import subprocess

assert 'google.colab' in sys.modules or __import__('importlib').util.find_spec('google.colab'), 'Notebook dành riêng cho Google Colab'

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
    if gpu.returncode == 0 and 'GPU ' in gpu.stdout:
        raise AssertionError('⚠️ CẢNH BÁO: Máy đang bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn CPU rồi chạy lại.')

print('✅ CHUẨN XÁC: Môi trường đang ở chế độ CPU-only (không có GPU accelerator).')
!lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\(s\)'

## 3. Clone Repository và Cài đặt thư viện
> Tự động clone mã nguồn từ GitHub, chuyển sang nhánh mới nhất và cài đặt các phụ thuộc `[reporting,dev]`.

In [ ]:
SOURCE_URL = 'https://github.com/LeoTkiet/arxiv-data-science.git'
# Ưu tiên nhánh main (hoặc có thể trỏ commit hash cụ thể)
SOURCE_REF = 'main'
SOURCE = Path('/content/arxiv-data-science')

if not SOURCE.exists():
    subprocess.run(['git', 'clone', SOURCE_URL, str(SOURCE)], check=True)
else:
    subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin'], check=True)

# Checkout ref mong muốn
try:
    subprocess.run(['git', '-C', str(SOURCE), 'checkout', SOURCE_REF], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'pull', 'origin', SOURCE_REF], check=True)
except Exception:
    # Nếu main chưa merge, tự động fallback về nhánh PR của TV5
    print('ℹ️ Fallback sang nhánh feature/tv5-ms1-benchmark-report...')
    subprocess.run(['git', '-C', str(SOURCE), 'checkout', 'feature/tv5-ms1-benchmark-report'], check=True)

commit = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'📌 Đang dùng commit: {commit}')

# Cài đặt pipeline ở chế độ reporting
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE) + '[reporting,dev]'], check=True)

# Kiểm tra test suite nhanh
import os
os.chdir(SOURCE)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/crawling', 'tests/test_validation.py', '-q'], check=True, cwd=str(SOURCE))


## 4. Cấu hình Semantic Scholar API Key (Tùy chọn)
> Đặt `S2_API_KEY` trong **Colab Secrets** (biểu tượng chìa khóa ở thanh công cụ bên trái). Nếu không có key, hệ thống vẫn chạy bình thường với pool công khai.

In [ ]:
try:
    from google.colab import userdata
    key = userdata.get('S2_API_KEY')
    if key:
        os.environ['S2_API_KEY'] = key
        print('✅ Đã nạp S2_API_KEY từ Colab Secrets.')
    else:
        print('ℹ️ Chưa đặt S2_API_KEY trong Colab Secrets (dùng rate limit công khai).')
except Exception:
    print('ℹ️ Bỏ qua Colab Secrets.')

## 5. Chọn Thành viên & Cấu hình Dải chạy (Interactive Form)
> Thành viên chọn đúng vai trò của mình. Dữ liệu sẽ được tự động định tuyến đến thư mục riêng trên Google Drive, **hoàn toàn độc lập và không sợ ghi đè lẫn nhau**.

In [ ]:
# @title 🎯 CHỌN PHẦN VIỆC CỦA BẠN { run: 'auto' }
MEMBER_ROLE = 'TV2' # @param ['TEST_SMOKE_1_BAI', 'TV1', 'TV2', 'TV3', 'TV4', 'TV5', 'ALL_GROUP_25K']
SAVE_TO_DRIVE = True # @param {type:'boolean'}
AUTO_RESUME = True # @param {type:'boolean'}
RUN_BENCHMARK = False # @param {type:'boolean'}

import datetime, json

RANGES_MAP = {
    'TEST_SMOKE_1_BAI': {'type': 'id', 'val': '2508.12872', 'folder': 'smoke_test'},
    'TV1': {'type': 'range', 'val': '2508.12872:2508.17871', 'folder': 'tv1'},
    'TV2': {'type': 'range', 'val': '2506.15788:2506.20787', 'folder': 'tv2'},
    'TV3': {'type': 'range', 'val': '2411.08799:2411.13798', 'folder': 'tv3'},
    'TV4': {'type': 'range', 'val': '2410.18025:2410.23024', 'folder': 'tv4'},
    'TV5': {'type': 'range', 'val': '2410.03025:2410.08024', 'folder': 'tv5'},
    'ALL_GROUP_25K': {'type': 'all', 'val': 'all', 'folder': 'all_group6'}
}

cfg = RANGES_MAP[MEMBER_ROLE]
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

base_root = Path('/content/drive/MyDrive/arxiv_ms1_data') if SAVE_TO_DRIVE else Path('/content/ms1-data')
OUTPUT = base_root / cfg['folder'] / 'data'
RESULTS = base_root / cfg['folder'] / 'artifacts' / f'run_{stamp}'

OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Chuẩn bị đối số dòng lệnh
assignment = json.loads((SOURCE / 'configs/ms1_group6_ranges.json').read_text())
if cfg['type'] == 'id':
    cli_args = ['--ids', cfg['val']]
    paper_count = 1
elif cfg['type'] == 'range':
    cli_args = ['--range', cfg['val']]
    paper_count = 5000
elif cfg['type'] == 'all':
    cli_args = [arg for r in assignment['ranges'] for arg in ['--range', r['range']]]
    paper_count = assignment['assigned_input_count']

if AUTO_RESUME:
    cli_args += ['--resume']

print('================ CẤU HÌNH LƯỢT CHẠY ===============')
print(f'👤 Thành viên: {MEMBER_ROLE}')
print(f'📦 Số bài dự kiến: {paper_count:,} bài')
print(f'💾 Thư mục lưu dữ liệu & checkpoint: {OUTPUT}')
print(f'📊 Thư mục lưu kết quả Benchmark: {RESULTS}')
print(f'🔄 Tự động khôi phục (Resume): {AUTO_RESUME}')
print(f'💽 Dung lượng đĩa còn trống: {shutil.disk_usage(OUTPUT).free / (1024**3):.1f} GB')
print('====================================================')
if not RUN_BENCHMARK:
    print('👉 HÃY ĐỔI RUN_BENCHMARK = True Ở FORM TRÊN KHI SẴN SÀNG CHẠY!')

## 6. Khởi chạy Pipeline & Benchmark
> Chạy thông qua module `benchmark.py` của TV5: Vừa crawl vừa liên tục lấy mẫu RAM, Disk, Wall-time chuẩn xác.  
> **Lưu ý:** Nếu phiên Colab bị ngắt sau vài tiếng, bạn chỉ cần chạy lại từ Cell 1 đến Cell 6 này, hệ thống sẽ tự động resume từ điểm dừng.

In [ ]:
assert RUN_BENCHMARK, 'Đổi RUN_BENCHMARK = True ở Cell 5 trước khi bấm chạy.'

command = [
    sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.benchmark',
    '--output', str(OUTPUT),
    '--results', str(RESULTS),
    '--', *cli_args
]

print('🚀 BẮT ĐẦU CHẠY PIPELINE...')
print('Lệnh thực thi:', ' '.join(command))

completed = subprocess.run(command, cwd=SOURCE)
print(f'🏁 PIPELINE ĐÃ HOÀN TẤT VỚI MÃ THOÁT: {completed.returncode}')

## 7. Thẩm định dữ liệu (Audit & Statistics)
> Tự động kiểm tra tính toàn vẹn của corpus và đối chiếu số lượng bài đã xử lý thành công so với dải 5.000 bài được giao.

In [ ]:
report_path = RESULTS / 'benchmark.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    report['environment']['is_colab'] = True
    report['environment']['cpu_only_confirmation'] = 'CPU-only verified: NVIDIA GPU probe passed'
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    
    # Chạy revalidate nếu có snapshot hiện tại
    if report.get('statistics_status') == 'current_run':
        subprocess.run([
            sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.statistics',
            '--output', str(OUTPUT),
            '--save', str(RESULTS / 'statistics-revalidated.json'),
            '--revalidate'
        ], cwd=SOURCE)
        if (RESULTS / 'statistics-revalidated.json').exists():
            stats = json.loads((RESULTS / 'statistics-revalidated.json').read_text(encoding='utf-8'))
            print('📈 KẾT QUẢ THỐNG KÊ (RATES):')
            for k, v in stats.get('rates', {}).items():
                print(f'  - {k}: {v.get("value", 0)*100:.2f}% ({v.get("numerator")}/{v.get("denominator")})')

# Chạy audit đối chiếu với toàn bộ dải bài được giao
audit_file = RESULTS / 'assignment-audit.json'
subprocess.run([
    sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.audit',
    '--output', str(OUTPUT),
    '--assignment', str(SOURCE / 'configs/ms1_group6_ranges.json'),
    '--save', str(audit_file)
], cwd=SOURCE)

if audit_file.exists():
    audit_data = json.loads(audit_file.read_text(encoding='utf-8'))
    print('📋 KẾT QUẢ KIỂM TOÁN DẢI ID (AUDIT):')
    print(f'  - Tổng ID đã phân giải (resolved): {audit_data.get("resolved_discovery_papers", 0)}')
    print(f'  - Số ID chưa xử lý (unseen): {audit_data.get("unseen_assigned_inputs_count", 0)}')
    print(f'  - Đạt chuẩn validation: {audit_data.get("validation_pass_papers", 0)} bài')

## 8. Đóng gói Artifacts và Tải về
> Nén toàn bộ tệp kết quả (`benchmark.json`, `statistics.json`, `samples.jsonl`, `timings.jsonl`, `assignment-audit.json`) thành tệp zip. Tệp này được lưu trên Google Drive VÀ tự động tải về máy qua trình duyệt để bạn gửi cho TV5 đưa vào Jupyter Book.

In [ ]:
from google.colab import files

archive_base = f'/content/TV_{cfg["folder"]}_MS1_RESULTS_{stamp}'
archive_path = shutil.make_archive(archive_base, 'zip', RESULTS)

# Copy 1 bản lưu trên Google Drive cho an toàn
if SAVE_TO_DRIVE:
    shutil.copy(archive_path, DRIVE_BASE / f'TV_{cfg["folder"]}_MS1_RESULTS_{stamp}.zip')
    print(f'💾 Đã lưu 1 bản zip trên Google Drive tại: {DRIVE_BASE}')

print(f'🎉 TỆP KẾT QUẢ SẴN SÀNG: {archive_path}')
print('Đang kích hoạt tải về máy...')
try:
    files.download(archive_path)
except Exception as e:
    print('ℹ️ Tải qua trình duyệt bị chặn, bạn có thể lấy trực tiếp file trên Google Drive:', e)

---
### 💡 MẸO GIỮ PHIÊN GOOGLE COLAB KHÔNG BỊ DISCONNECT (KEEP-ALIVE)
Để tránh Colab tự ngắt khi chạy tác vụ dài 10-12 tiếng:
1. Nhấn phím `F12` (hoặc `Ctrl + Shift + I` trên Windows, `Cmd + Option + I` trên Mac) trên trình duyệt để mở **Developer Console**.
2. Chọn thẻ **Console**.
3. Dán đoạn mã sau vào và nhấn `Enter`:
```javascript
function ClickConnect(){
    console.log("Colab Keep-Alive: Dang giu ket noi...");
    document.querySelector("colab-connect-button")?.click() || document.querySelector("#connect")?.click();
}
setInterval(ClickConnect, 60000); // 1 phut click 1 lan
```